In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
from tqdm import tqdm
import datetime as dt

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-03-06 07:46:45.271686


### Functions

In [3]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [4]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')
# output
str_dirname_output = './output'
# dict
list_str_replace = [
    '1_in_30',
    '1_in_60',
    '1_in_90',
    '1_in_120',
    '1_in_150',
    '1_in_180',
    '1_in_210',
    '1_in_240',
    '1_in_270',
    '1_in_300',
    '1_in_330',
    '1_in_360',
    '1_in_390',
    '1_in_420',
    '1_in_450',
    '1_in_480',
    '1_in_510',
    '1_in_540',
    '1_in_570',
    '1_in_600',
    '1_in_630',
    '1_in_660',
    '1_in_690',
    '1_in_720',
    '15_in_30',
    '15_in_60',
    '15_in_90',
    '15_in_120',
    '15_in_150',
    '15_in_180',
    '15_in_210',
    '15_in_240',
    '15_in_270',
    '15_in_300',
    '15_in_330',
    '15_in_360',
    '15_in_390',
    '15_in_420',
    '15_in_450',
    '15_in_480',
    '15_in_510',
    '15_in_540',
    '15_in_570',
    '15_in_600',
    '15_in_630',
    '15_in_660',
    '15_in_690',
    '15_in_720',
    '30_in_30',
    '30_in_60',
    '30_in_90',
    '30_in_120',
    '30_in_150',
    '30_in_180',
    '30_in_210',
    '30_in_240',
    '30_in_270',
    '30_in_300',
    '30_in_330',
    '30_in_360',
    '30_in_390',
    '30_in_420',
    '30_in_450',
    '30_in_480',
    '30_in_510',
    '30_in_540',
    '30_in_570',
    '30_in_600',
    '30_in_630',
    '30_in_660',
    '30_in_690',
    '30_in_720',
    '60_in_90',
    '60_in_120',
    '60_in_150',
    '60_in_180',
    '60_in_210',
    '60_in_240',
    '60_in_270',
    '60_in_300',
    '60_in_330',
    '60_in_360',
    '60_in_390',
    '60_in_420',
    '60_in_450',
    '60_in_480',
    '60_in_510',
    '60_in_540',
    '60_in_570',
    '60_in_600',
    '60_in_630',
    '60_in_660',
    '60_in_690',
    '60_in_720',
    '90_in_120',
    '90_in_150',
    '90_in_180',
    '90_in_210',
    '90_in_240',
    '90_in_270',
    '90_in_300',
    '90_in_330',
    '90_in_360',
    '90_in_390',
    '90_in_420',
    '90_in_450',
    '90_in_480',
    '90_in_510',
    '90_in_540',
    '90_in_570',
    '90_in_600',
    '90_in_630',
    '90_in_660',
    '90_in_690',
    '90_in_720',
]

Project: 20231010-gen-xii
Task: 09_early_indicators
Subtask: 05_get_target_from_db


### Output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### Read query

In [6]:
str_filepath = './sql/query.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('SELECT \n'
 '\tam.bigAccountId,\n'
 '\tCASE WHEN Early_Pay_Delinquency_STRDPD_STRTOTAL > 0 THEN 1 ELSE 0 END AS '
 'Early_Pay_Delinquency_STRDPD_STRTOTAL_Flag\n'
 'FROM \n'
 '\telectra.pfsdb.dbo.tblAccountMaintenance am LEFT OUTER JOIN\n'
 '\t(\n'
 '\t\tSELECT \n'
 '\t\t\tc.bigAccountId,\n'
 '\t\t\tSUM(CASE\n'
 '\t\t\t\t\tWHEN DATEDIFF(DAY, c.dtmDue, c.dtmClosed) > DPDMINUS1 OR '
 '(DATEDIFF(DAY, c.dtmDue, GETDATE()) > DPDMINUS1 AND c.dtmClosed IS NULL) \n'
 '\t\t\t\t\tTHEN 1\n'
 '\t\t\t\t\tELSE 0\n'
 '\t\t\t\tEND) AS Early_Pay_Delinquency_STRDPD_STRTOTAL\n'
 '\t\tFROM pfsdb.dbo.tblCharges c LEFT OUTER JOIN pfsdb.dbo.tblAccountTerms '
 'att ON c.bigAccountTermId = att.bigAccountTermId\n'
 '\t\t\tINNER JOIN pfsdb.dbo.tblAccountMaintenance am ON c.bigAccountId = '
 'am.bigAccountId\n'
 '\t\tWHERE c.bigChargeTypeId = 1 AND c.bitInvalid = 0 AND c.dtmDue < '
 'DATEADD(DAY, STRTOTAL, am.dtmContract)\n'
 '\t\tGROUP BY c.bigAccountId\n'
 '\t)a\n'
 '\tON am.bigAccountId = a.bigAccountId')


### DB Connection

In [7]:
# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)

### Iterate, replace, pull, and join

In [8]:
list_df = []
for str_replace in tqdm(list_str_replace):
    # DPD
    STRDPD = str_replace.split('_in_')[0]
    # total
    STRTOTAL = str_replace.split('_in_')[1]
    # DPD minus 1
    STRDPDMINUS1 = str(int(STRDPD) - 1)
    
    # replace
    str_query_tmp = str_query.replace('STRDPD', STRDPD)
    str_query_tmp = str_query_tmp.replace('STRTOTAL', STRTOTAL)
    str_query_tmp = str_query_tmp.replace('DPDMINUS1', STRDPDMINUS1)
    
    # pull from db
    df = pd.read_sql_query(
        str_query_tmp, 
        con=conn,
    )
    # set index
    df = df.set_index('bigAccountId')
    # append
    list_df.append(df)

# close
conn.close()

# join
df = pd.concat(list_df, axis=1, join='outer')
# reset index
df.reset_index(inplace=True)
# show
df

100%|████████████████████████████████████████████████████████████████████████████████| 115/115 [40:46<00:00, 21.28s/it]


,bigAccountId,Early_Pay_Delinquency_1_30_Flag,Early_Pay_Delinquency_1_60_Flag,Early_Pay_Delinquency_1_90_Flag,Early_Pay_Delinquency_1_120_Flag,Early_Pay_Delinquency_1_150_Flag,Early_Pay_Delinquency_1_180_Flag,Early_Pay_Delinquency_1_210_Flag,Early_Pay_Delinquency_1_240_Flag,Early_Pay_Delinquency_1_270_Flag,...,Early_Pay_Delinquency_90_450_Flag,Early_Pay_Delinquency_90_480_Flag,Early_Pay_Delinquency_90_510_Flag,Early_Pay_Delinquency_90_540_Flag,Early_Pay_Delinquency_90_570_Flag,Early_Pay_Delinquency_90_600_Flag,Early_Pay_Delinquency_90_630_Flag,Early_Pay_Delinquency_90_660_Flag,Early_Pay_Delinquency_90_690_Flag,Early_Pay_Delinquency_90_720_Flag
0,6,0,0,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,25,0,1,1,1,1,1,1,1,1,...,0,1,1,1,1,1,1,1,1,1
2,73,0,0,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,1,1,1
3,82,0,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
4,122,0,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
326637,7630411,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
326638,7630777,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
326639,7630821,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
326640,7631084,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Save

In [9]:
%%time

# save
str_filename = 'df_early_targets.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

Wall time: 15.1 s


### Upload to s3

In [10]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_subtask}/{str_filename}', 
    str_bucket_name=str_project,
)

Wall time: 2.23 s


### Clean-up

In [11]:
os.remove(str_local_path)